# Exercise 11 : An Async Subagent on a Server

- Co-deployed graphs share one Agent Server ; the parent starts child work through async task tools
- Lifecycle tools start, check, list and collect ; the parent copies full task ids from state
- Local `langgraph dev` needs enough `--n-jobs-per-worker` for parent and child runs

You will serve an Aurelia supervisor and a diagnosis worker, start a delayed
diagnosis as an async subagent, check it, list it, collect the result, and stop
the server. Laptop only : Colab cannot start the Agent Server. All data is made up.

Run the next cell first. Its code is folded : click the cell and press Shift+Enter. It loads the course key and builds `llm`, the course model.

In [1]:
#@title Course setup : run this cell first
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## Step 1 : Graphs and `langgraph.json`

A server starts from files : the graph code and a `langgraph.json` that lists
the graphs to serve (slide *Co-deployed Graphs*). Both graphs go on one server
here, so the supervisor can start the worker through async task tools. There is
nothing to write. The first cell, with its code folded, writes
`course_runtime.py`, the setup code as a file the server can import. The second
cell has the source of `graphs.py` as the string `graphs_source`, and writes
`graphs.py` and `langgraph.json` next to the notebook.

Read `graphs_source` in the cell. `diagnosis` is a graph with one node,
`diagnose`, which waits 120 s unless the task text contains `NO DELAY` and then
runs `worker`. `supervisor` is a Deep Agent that has `diagnosis` as an
`AsyncSubAgent`. Run both cells and read the printed configuration, then run the
file check. No server runs yet and this step makes no model call. A failed check
means a cell above it did not finish, so read its error first.

In [2]:
from pathlib import Path
Path('course_runtime.py').write_text('#@title Course setup : run this cell first\n# On the classroom laptop the key comes from .env in the repository root.\n# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.\nimport os\nimport sys\nimport subprocess\nimport warnings\nfrom pathlib import Path\n\nIN_COLAB = "google.colab" in sys.modules\nCOURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"\nROOT = None\n\nif IN_COLAB:\n    # Pinned to match uv.lock ; generated by build/stamp_setup.py\n    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]\n    if not os.getenv("COURSE_SKIP_INSTALL"):\n        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)\n    try:\n        from google.colab import userdata\n        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))\n    except Exception:\n        pass\nelse:\n    from dotenv import load_dotenv\n    for folder in [Path.cwd(), *Path.cwd().parents]:\n        if (folder / "pyproject.toml").exists():\n            ROOT = folder\n            load_dotenv(folder / ".env")\n            break\n\nPROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"\nif PROVIDER == "ollama":\n    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")\n    BASE_URL = "http://localhost:11434/v1"\n    API_KEY = "ollama"\n    THINKING_OFF = {"reasoning_effort": "none"}\nelse:\n    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up\n    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")\n    API_KEY = os.getenv("COURSE_API_KEY")\n    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;\n    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider\n    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}\n    if not API_KEY:\n        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")\n\nfrom langchain_core._api import LangChainBetaWarning\nwarnings.filterwarnings("ignore", category=LangChainBetaWarning)\n\nfrom langchain_openai import ChatOpenAI\nfrom langchain_core.rate_limiters import InMemoryRateLimiter\n\n# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.\n# Paid OpenRouter models have no platform request cap (measured 26 October 2026)\nRATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),\n                                   check_every_n_seconds=0.1, max_bucket_size=1)\n\n\ndef make_llm(**kw):\n    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""\n    kw.setdefault("model", MODEL)\n    if PROVIDER == "ollama":\n        kw.setdefault("temperature", 0)\n    else:\n        kw.setdefault("seed", 7)\n    kw.setdefault("extra_body", THINKING_OFF)\n    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging\n    kw.setdefault("rate_limiter", RATE_LIMITER)\n    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)\n\n\ndef make_embeddings(**kw):\n    """Local sentence embeddings, so retrieval costs no API calls."""\n    from langchain_huggingface import HuggingFaceEmbeddings\n    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")\n    return HuggingFaceEmbeddings(**kw)\n\n\ndef data(name):\n    """Path to a course data file ; downloaded when there is no local copy."""\n    if ROOT and (ROOT / "data" / name).exists():\n        return str(ROOT / "data" / name)\n    target = Path(name)\n    if not target.exists():\n        import urllib.request\n        target.parent.mkdir(parents=True, exist_ok=True)\n        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)\n    return str(target)\n\n\ndef data_folder(name):\n    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""\n    if ROOT and (ROOT / "data" / name).is_dir():\n        return str(ROOT / "data" / name)\n    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():\n        data(f"{name}/{file}")\n    return name\n\n\nllm = make_llm()\nprint(f"model {MODEL} via {PROVIDER}")\n', encoding='utf-8')
print('wrote course_runtime.py')

wrote course_runtime.py


In [3]:
graphs_source = r'''
"""Aurelia diagnosis graphs for exercise 11."""
import asyncio
import json
import sqlite3
from pathlib import Path

from course_runtime import data, data_folder, llm
from deepagents import AsyncSubAgent, create_deep_agent
from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.graph import END, START, MessagesState, StateGraph

DB = Path(data("capstone/service.db")).resolve()
MANUALS = Path(data_folder("capstone/manuals"))


@tool
def part_stock(part_no: str) -> str:
    """Look up an Aurelia part's stock and depot by its part number."""
    with sqlite3.connect(f"file:{DB.as_posix()}?mode=ro", uri=True) as db:
        rows = db.execute("SELECT part_no, stock, depot FROM parts WHERE part_no = ?",
                          (part_no.upper(),)).fetchall()
    return json.dumps(rows)


@tool
def fault_guidance(error_code: str) -> str:
    """Read the meaning and first action for an Aurelia error code."""
    return "\n".join(f"[{p.name}] {line}" for p in sorted(MANUALS.glob("*.md"))
                     for line in p.read_text(encoding="utf-8").splitlines()
                     if line.startswith("| " + error_code.upper() + " |")) or "No matching code."


SYSTEM = ("Support Aurelia field engineers. Use tools for facts. Preserve stock, "
          "depot and first actions. Be brief.")
worker = create_agent(llm, tools=[part_stock, fault_guidance], system_prompt=SYSTEM, name="worker")


async def diagnose(state):
    text = str(state["messages"][-1].content)
    # Delay rule : every run waits 120 s, unless the task text contains NO DELAY.
    # The wait stands in for slow work, so the first check finds the task running.
    if "NO DELAY" not in text:
        await asyncio.sleep(120)
    result = await worker.ainvoke({"messages": state["messages"]})
    return {"messages": result["messages"]}


builder = StateGraph(MessagesState)
builder.add_node("diagnose", diagnose)
builder.add_edge(START, "diagnose")
builder.add_edge("diagnose", END)
diagnosis = builder.compile()

supervisor = create_deep_agent(
    model=llm,
    subagents=[AsyncSubAgent(name="diagnosis", graph_id="diagnosis",
                             description="Long Aurelia fault diagnosis with manuals and parts.")],
    system_prompt=(
        "Coordinate Aurelia diagnosis. Use only the requested async task tool, then "
        "respond briefly. After launch return to the user without checking. Copy full "
        "task IDs from async_tasks. Always fetch status before reporting it."
    ),
    name="supervisor",
)
'''

import json

Path("graphs.py").write_text(graphs_source.lstrip(), encoding="utf-8")
config = {"dependencies": ["."],
          "graphs": {"diagnosis": "./graphs.py:diagnosis", "supervisor": "./graphs.py:supervisor"},
          "env": "../.env"}
Path("langgraph.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("wrote graphs.py, langgraph.json")
print(json.dumps(config, indent=2))

wrote graphs.py, langgraph.json
{
  "dependencies": [
    "."
  ],
  "graphs": {
    "diagnosis": "./graphs.py:diagnosis",
    "supervisor": "./graphs.py:supervisor"
  },
  "env": "../.env"
}


In [4]:
print("check :", "PASS" if Path("graphs.py").exists() and Path("langgraph.json").exists() else "FAIL")
assert Path("graphs.py").exists() and Path("langgraph.json").exists()

check : PASS


## Step 2 : Start the server

`langgraph dev` runs a local Agent Server for the graphs in `langgraph.json`
(slide *langgraph dev and Studio*). The first cell starts nothing. It defines
helpers : `start_server` launches `langgraph dev` and waits for `/ok`, `connect`
returns a client, `stop_server` stops a server, and `clear_dev_storage` removes
the storage of an earlier run. `start_server` passes `--n-jobs-per-worker 8`, so
the parent run and the child run can work at once.

Run the first cell, then replace the two blanks in the second cell. Both blanks
take the same port number, without quotes, and the `/ok` line in the cell shows
it.

A good run prints `ready :` with `ok` True and `assistants :` with one entry per
graph, and the check prints PASS. A `ValueError` means a port outside this
exercise. A `TimeoutError` means the port stayed busy, usually because a server
from an earlier run or a demo is still up. If the first run of this cell started
a server, call `stop_server(server)` before you run the cell again. A
`RuntimeError` prints the end of `server-2134.log` : the server stopped during
startup, and the log gives the reason.

In [5]:
import atexit
import os
import signal
import socket
import subprocess
import time
from pathlib import Path

import httpx
from langgraph_sdk import get_sync_client

HERE = Path.cwd()
servers = []


def port_open(port):
    with socket.socket() as probe:
        probe.settimeout(0.5)
        return probe.connect_ex(("127.0.0.1", port)) == 0


def wait_port_free(port, seconds=60):
    for _ in range(seconds * 2):
        if not port_open(port):
            return
        time.sleep(0.5)
    raise TimeoutError(f"Port {port} stayed busy after stop.")


def stop_server(proc):
    if proc is None:
        return
    port = getattr(proc, "course_port", None)
    was_running = proc.poll() is None
    if was_running:
        proc.send_signal(signal.CTRL_BREAK_EVENT if os.name == "nt" else signal.SIGINT)
        try:
            proc.wait(timeout=25)
        except subprocess.TimeoutExpired:
            if os.name == "nt":
                subprocess.run(["taskkill", "/PID", str(proc.pid), "/T", "/F"],
                               capture_output=True, check=False)
            else:
                proc.kill()
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                pass
    if was_running and port is not None:
        wait_port_free(port, seconds=90)


def start_server(port):
    if port not in (2134, 2135):
        raise ValueError("Use an exercises 08-11 port (2134 or 2135).")
    wait_port_free(port, seconds=90)
    log = (HERE / f"server-{port}.log").open("w", encoding="utf-8")
    env = dict(os.environ, PYTHONIOENCODING="utf-8", LANGGRAPH_CLI_NO_ANALYTICS="1")
    env["LANGSMITH_TRACING"] = "false"
    proc = subprocess.Popen(
        ["uv", "run", "langgraph", "dev", "--no-browser", "--no-reload", "--port", str(port),
         "--n-jobs-per-worker", "8", "--studio-url", "https://eu.smith.langchain.com"],
        cwd=HERE, env=env, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NEW_PROCESS_GROUP if os.name == "nt" else 0)
    proc.course_port = port
    log.close()
    servers.append(proc)
    atexit.register(stop_server, proc)
    for _ in range(180):
        if proc.poll() is not None:
            raise RuntimeError((HERE / f"server-{port}.log").read_text(encoding="utf-8")[-5000:])
        try:
            response = httpx.get(f"http://127.0.0.1:{port}/ok", timeout=1)
            if response.is_success:
                return proc
        except httpx.TransportError:
            pass
        time.sleep(0.5)
    stop_server(proc)
    raise TimeoutError("Agent Server did not become ready.")


def connect(port):
    return get_sync_client(url=f"http://127.0.0.1:{port}", api_key=None, timeout=240)


def clear_dev_storage():
    import shutil
    target = HERE / ".langgraph_api"
    if target.exists():
        shutil.rmtree(target)
        print("cleared :", target.name)

In [6]:
clear_dev_storage()
server = start_server(2134)
client = connect(2134)
task_id = None  # a new server has no task yet
print("ready :", httpx.get("http://127.0.0.1:2134/ok").json())
print("assistants :", [(a["name"], a["graph_id"]) for a in client.assistants.search()])

cleared : .langgraph_api


ready : {'ok': True, 'services': {}}
assistants : [('supervisor', 'supervisor'), ('diagnosis', 'diagnosis')]


In [7]:
names = {a["graph_id"] for a in client.assistants.search()}
print("check :", "PASS" if {"diagnosis", "supervisor"} <= names else "FAIL", names)
assert {"diagnosis", "supervisor"} <= names

check : PASS {'diagnosis', 'supervisor'}


## Step 3 : Start, check and list

An async subagent works in the background while its parent stays available
(slide *Async Subagents*). `supervise` sends one message to the `supervisor`
graph on thread `tid`, prints the `response` and `async_tasks`, and adds every
tool the model called to `used`. Its `client.runs.wait` returns when the
supervisor finishes its reply, and the worker keeps running. The cell then asks
for three lifecycle tools in turn : `start_async_task`, `check_async_task` and
`list_async_tasks`. The task text does not contain `NO DELAY`, so the worker
waits 120 s before its model call, and the first check should run while the task
is still `running`.

Replace the `____` on the last line, then run the cell. Write assertions that
`first_status` has the expected status and that `used` includes all three
requested tools. Each run of the cell starts a new task. The first lines of the
cell therefore cancel the task of an earlier run, so no second task keeps
working.

The output shows the `task ID`, then `first check status`, then `lifecycle so
far` with the tool names. An error on the line that sets `task_id` means the
supervisor started no task ; read `response` and run the cell again. A failed
assertion on `used` means the model replied without calling a tool, so run the
cell again. Step 4 needs `out` and `task_id` from this cell.

In [8]:
# A re-run of this cell starts a new task, so it first cancels the task of the earlier run.
if task_id:
    for run in client.runs.list(task_id):
        if run["status"] in ("pending", "running"):
            client.runs.cancel(task_id, run["run_id"])
    print("cancelled earlier task :", task_id)

tid = client.threads.create()["thread_id"]
used = set()


def supervise(text):
    result = client.runs.wait(tid, "supervisor", input={"messages": [{"role": "user", "content": text}]},
                              config={"recursion_limit": 30}, raise_error=True)
    if "messages" not in result:
        raise RuntimeError("supervisor run finished without messages : " + repr(result)[:800])
    print("response :", result["messages"][-1]["content"])
    print("tasks :", result.get("async_tasks", {}))
    used.update(c["name"] for m in result["messages"] for c in m.get("tool_calls", []) or [])
    return result


# This task text has no NO DELAY, so the worker waits 120 s before it starts.
out = supervise(
    "Use start_async_task for diagnosis. Task: diagnose E-CT-110 and look up "
    "CT-FN-02 stock. Return the full task ID and stop. Do not check yet."
)
task_id = next(iter(out["async_tasks"]))
print("task ID :", task_id)
out = supervise(f"Use check_async_task for {task_id}. Report its fresh status and stop.")
first_status = out["async_tasks"][task_id]["status"]
print("first check status :", first_status)
out = supervise("Use list_async_tasks with status_filter all. Report the current list and stop.")
print("lifecycle so far :", sorted(used))
assert {"start_async_task", "check_async_task", "list_async_tasks"} <= used
assert first_status == "running"

response : Task started: `01a196d3-bd87-7f32-8b90-bb569e3b06bd`
tasks : {'01a196d3-bd87-7f32-8b90-bb569e3b06bd': {'task_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'agent_name': 'diagnosis', 'thread_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'run_id': '01a196d3-bd88-7e52-a460-5ecd3aeef5f6', 'status': 'running', 'created_at': '2026-11-01T11:17:01Z', 'last_checked_at': '2026-11-01T11:17:04Z', 'last_updated_at': '2026-11-01T11:17:04Z'}}
task ID : 01a196d3-bd87-7f32-8b90-bb569e3b06bd


response : Fresh status: **running**.
tasks : {'01a196d3-bd87-7f32-8b90-bb569e3b06bd': {'task_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'agent_name': 'diagnosis', 'thread_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'run_id': '01a196d3-bd88-7e52-a460-5ecd3aeef5f6', 'status': 'running', 'created_at': '2026-11-01T11:17:01Z', 'last_checked_at': '2026-11-01T11:17:06Z', 'last_updated_at': '2026-11-01T11:17:04Z'}}
first check status : running


response : Tracked tasks (all): `01a196d3-bd87-7f32-8b90-bb569e3b06bd` — **running**.
tasks : {'01a196d3-bd87-7f32-8b90-bb569e3b06bd': {'task_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'agent_name': 'diagnosis', 'thread_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'run_id': '01a196d3-bd88-7e52-a460-5ecd3aeef5f6', 'status': 'running', 'created_at': '2026-11-01T11:17:01Z', 'last_checked_at': '2026-11-01T11:17:10Z', 'last_updated_at': '2026-11-01T11:17:04Z'}}
lifecycle so far : ['check_async_task', 'list_async_tasks', 'start_async_task']


In [9]:
print("first check status :", first_status)
print("check :", "PASS" if first_status == "running" and {"start_async_task", "check_async_task", "list_async_tasks"} <= used else "FAIL")
assert first_status == "running"
assert {"start_async_task", "check_async_task", "list_async_tasks"} <= used

first check status : running
check : PASS


## Step 4 : Collect the result

The last lifecycle step collects the result with a fresh check after the child
run has ended (slide *Async Task Lifecycle*). `task` is this task's entry in
`async_tasks`, and `client.runs.join(task_id, task["run_id"])` waits for the
child run. That call works because the task id is the worker's thread id. The
cell stays quiet while the worker finishes its wait and its diagnosis, and that
wait is expected. `supervise` then checks again, `status` is the final status,
and `text` is the answer in lower case.

Replace the `____` on the last line with assertions on `status` and on `text`.
`status` must be the finished status that the demo asserts, and `text` must
contain the stock count, the depot and the part name for CT-FN-02. Write those
words in lower case. The `answer text` line prints before the blank raises, so
you can read the values there and run the cell again ; the second run does not
wait for the worker.

Run the final cell to stop every process in `servers`, and look for `servers
stopped : True`. Run it also when an assertion failed, because it stops the
servers before it checks anything. A failed assertion on `text` usually means
the supervisor reworded the answer or dropped a fact ; read `answer text` and
run the cell again.

In [10]:
task = out["async_tasks"][task_id]
client.runs.join(task_id, task["run_id"])
out = supervise(f"Use check_async_task for {task_id}. Return the result, first action and stock.")
status = out["async_tasks"][task_id]["status"]
print("final status :", status)
text = str(out["messages"][-1]["content"]).lower()
print("answer text :", text[:500])
assert status == "success"
assert "6" in text and "houten" in text and "cooling fan" in text

response : E-CT-110 indicates gantry over-temperature. **First action:** check the gantry cooling fan; replace it if airflow is below 80%. **CT-FN-02 stock:** 6 units at the Houten depot.
tasks : {'01a196d3-bd87-7f32-8b90-bb569e3b06bd': {'task_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'agent_name': 'diagnosis', 'thread_id': '01a196d3-bd87-7f32-8b90-bb569e3b06bd', 'run_id': '01a196d3-bd88-7e52-a460-5ecd3aeef5f6', 'status': 'success', 'created_at': '2026-11-01T11:17:01Z', 'last_checked_at': '2026-11-01T11:19:08Z', 'last_updated_at': '2026-11-01T11:19:06Z'}}
final status : success
answer text : e-ct-110 indicates gantry over-temperature. **first action:** check the gantry cooling fan; replace it if airflow is below 80%. **ct-fn-02 stock:** 6 units at the houten depot.


In [11]:
for proc in servers:
    stop_server(proc)
print("servers stopped :", all(proc.poll() is not None for proc in servers))
print("check :", "PASS" if status == "success" else "FAIL")
assert status == "success"

servers stopped : True
check : PASS


## If time permits

- Cancel a second delayed task before it finishes
- Steer the running task with `update_async_task` to look up UX-BT-02 instead. The
  update starts a new run, which waits 120 s again unless its text contains `NO DELAY`
- Restart the server in the same folder and read whether the thread still lists the task